# Projeto 3: Análise de dados de um e-commerce (versão didática)

Este notebook faz o projeto **passo a passo**, com código simples e comentado.

**Como usar:**
1. Abra no **Google Colab** ou no **Jupyter**.
2. Coloque o arquivo `data.csv` na mesma pasta do notebook (no Colab, use o ícone de pasta à esquerda e faça o upload).
3. Rode as células **em ordem**, de cima para baixo (`Shift + Enter`).

**Roteiro**
- Passo 0: Importar as bibliotecas
- Passo 1: Conhecer os dados
- Passo 2: Limpar e preparar os dados
- Passo 3: Análise exploratória (EDA)
- Passo 4: Segmentação de clientes (RFM + K-Means)
- Passo 5: Conclusões

## Passo 0: Importar as bibliotecas

Bibliotecas são "caixas de ferramentas" prontas:
- **pandas**: trabalha com tabelas (como um Excel dentro do Python)
- **numpy**: faz contas
- **matplotlib** e **seaborn**: desenham gráficos
- **scikit-learn**: tem o algoritmo K-Means
- **mlxtend**: encontra produtos comprados juntos

A linha `!pip install mlxtend` instala a única biblioteca que normalmente não vem instalada.

In [1]:
!pip install mlxtend -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from mlxtend.frequent_patterns import apriori, association_rules

# Deixa os gráficos com um visual mais bonito
sns.set_theme(style='whitegrid')

# Mostra números com 2 casas decimais
pd.set_option('display.float_format', '{:,.2f}'.format)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Passo 1: Conhecer os dados

Vamos ler o arquivo e olhar as primeiras linhas.

> O arquivo usa acentos do padrão europeu, por isso usamos `encoding='ISO-8859-1'`.
> Lemos `CustomerID` como texto (`str`) porque é um código, não um número para fazer contas.

In [ ]:
df = pd.read_csv('/content/data.csv', encoding='ISO-8859-1', dtype={'CustomerID': str, 'InvoiceNo': str})

print('Linhas e colunas:', df.shape)
df.head()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

FileNotFoundError: [Errno 2] No such file or directory: '/content/data.csv'

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## **O que significa cada coluna:**


 **InvoiceNo** : Número da fatura (pedido). Se começa com **C**, é um **cancelamento**

**StockCode** : Código do produto

**Description**: Nome do produto

**Quantity**: Quantidade comprada

 **InvoiceDate** : Data e hora da compra

**UnitPrice**: Preço de uma unidade (em libras **£**)

**CustomerID** : Código do cliente

**Country**: País do cliente


In [ ]:
# Tipos de dado de cada coluna, a quantidade de linhas e colunas, valores preenchidos, e de Dataframe usado.
df.info()

In [ ]:
# Resumo estatístico das colunas numéricas
df.describe()

Existem **quantidades negativas** e **preços negativos**, Significa que existe algum problema nos dados.

In [ ]:
# Os valores que estão faltando em cada coluna.
df.isnull().sum()

In [ ]:
#Verificando as linhas duplicadas, os cancelamentos, a quatidade negativa e com preço menor ou igual a 0.
print('Linhas duplicadas:', df.duplicated().sum())
print('Linhas de cancelamento (fatura começa com C):', df['InvoiceNo'].str.startswith('C').sum())
print('Linhas com quantidade negativa:', (df['Quantity'] < 0).sum())
print('Linhas com preço zero ou negativo:', (df['UnitPrice'] <= 0).sum())

**Problemas encontrados:**
1. Cerca de **25% das linhas não têm CustomerID**. Dificulta a analise do perfil do cliente.
2. Há **linhas duplicadas**.
3. Há **cancelamentos** (quantidade negativa).
4. Há **preços zerados** e **ajustes de estoque** (quantidade negativa sem ser cancelamento).

## Passo 2: Limpar e preparar os dados

### 2.1 Remover duplicados

In [ ]:
df = df.drop_duplicates()
print('Linhas depois de remover duplicados:', len(df))

### 2.2 Converter a data

A data veio como texto. Convertendo para o tipo *data*, conseguimos extrair mês, dia da semana e hora.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate'].head()

### 2.3 Marcar os cancelamentos

Criamos uma coluna `Cancelamento` com **True** (verdadeiro) quando a fatura começa com "C".

In [ ]:
df['Cancelamento'] = df['InvoiceNo'].str.startswith('C')
df['Cancelamento'].value_counts()

### 2.4 Remover linhas que não são vendas de verdade

- Preço **zero ou negativo** podem ser brindes, erros ou ajustes de dividas
- Quantidade negativa **sem** ser cancelamento pode ser estoque ou devolução de produtos com defeito

In [ ]:
preco_valido = df['UnitPrice'] > 0
ajuste_estoque = (df['Quantity'] < 0) & (df['Cancelamento'] == False)

df = df[preco_valido & ~ajuste_estoque]   # o símbolo ~ significa "NÃO"
print('Nova quantidade de linhas:', len(df))

### 2.5 Criar novas colunas
A criação de novas colunas serve para ajudar entender o perfil das pessoas que compram no e-commerce.
- **Total** = Quantidade × Preço (valor de cada linha)
- **Mes**, **DiaSemana** e **Hora**: para estudar quando as pessoas compram

In [ ]:
df['Total'] = df['Quantity'] * df['UnitPrice']
df['AnoMes'] = df['InvoiceDate'].dt.to_period('M').astype(str)   # ex.: '2011-11'
df['DiaSemana'] = df['InvoiceDate'].dt.dayofweek                 # 0 = segunda ... 6 = domingo
df['Hora'] = df['InvoiceDate'].dt.hour

df.head()

In [ ]:
#Visulizando as ultimas linhas da tabela.
df.tail(10)

### 2.6 Separar vendas e cancelamentos

Aqui vamos separar e ver a quatidade de linhas da coluna vendas e Cancelamento.

In [ ]:
vendas = df[df['Cancelamento'] == False]
cancelamentos = df[df['Cancelamento'] == True]

print('Linhas de vendas:', len(vendas))
print('Linhas de cancelamento:', len(cancelamentos))

### 2.7 Métricas de sucesso (o ponto de partida)

- **Receita**: soma de todas as vendas
- **Ticket médio**: receita ÷ número de pedidos
- **Taxa de cancelamento**: faturas canceladas ÷ total de faturas
- **Recorrência**: % de clientes que compraram 2 vezes ou mais

In [ ]:
receita = vendas['Total'].sum()
valor_cancelado = -cancelamentos['Total'].sum()      # o sinal de menos deixa o valor positivo

pedidos = vendas['InvoiceNo'].nunique()              # nunique = conta valores diferentes
faturas_canceladas = cancelamentos['InvoiceNo'].nunique()

ticket_medio = receita / pedidos
taxa_cancelamento = faturas_canceladas / (pedidos + faturas_canceladas) * 100

pedidos_por_cliente = vendas.dropna(subset=['CustomerID']).groupby('CustomerID')['InvoiceNo'].nunique()
recorrencia = (pedidos_por_cliente >= 2).mean() * 100

print(f'Receita total:          £ {receita:,.2f}')
print(f'Valor cancelado:        £ {valor_cancelado:,.2f}')
print(f'Número de pedidos:      {pedidos}')
print(f'Ticket médio:           £ {ticket_medio:,.2f}')
print(f'Taxa de cancelamento:   {taxa_cancelamento:.1f}% das faturas')
print(f'Recorrência:            {recorrencia:.1f}% dos clientes compraram 2+ vezes')

## Passo 3: Análise exploratória (EDA)

### 3.1 Vendas ao longo do tempo

**groupby** foi usado para agrupar a coluna mes e ano e **sum**  para somar os valores de cada grupo.

In [ ]:
receita_mensal = vendas.groupby('AnoMes')['Total'].sum()

receita_mensal.plot(
    kind='area',
    figsize=(12, 4),
    color='cornflowerblue',
    alpha=0.5
)
plt.title('Volume da Receita Mensal (£)')
plt.xlabel('Mês')
plt.ylabel('Receita (£)')
plt.show()

In [ ]:
receita_mensal.plot(
    kind='line',
    figsize=(12, 4),
    color='seagreen',
    marker='o'
)
plt.title('Evolução da Receita Mensal (£)')
plt.xlabel('Mês')
plt.ylabel('Receita (£)')
plt.grid(True, alpha=0.1)
plt.show()

**O que vemos:** as vendas sobem muito de **setembro a novembro** (compras de Natal). Dezembro/2011 aparece baixo porque os dados terminam no dia 09/12, com isso não consegue analisar as vendas do mês inteiro.


**Analisando o Dia e a Hora de compra para ver o qual horario e dia que tem maior venda concentrada:**

In [ ]:
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
receita_dia = vendas.groupby('DiaSemana')['Total'].sum()
pedidos_hora = vendas.groupby('Hora')['InvoiceNo'].nunique()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))   # 1 linha e 2 gráficos lado a lado

ax[0].bar([dias[d] for d in receita_dia.index], receita_dia.values, color='steelblue')
ax[0].set_title('Receita por dia da semana')

ax[1].bar(pedidos_hora.index, pedidos_hora.values, color='coral')
ax[1].set_title('Pedidos por hora do dia')
ax[1].set_xlabel('Hora')

plt.show()

As vendas concentram-se nos dias úteis, sem registros aos sábados e com menor volume aos domingos. O período entre 10h e 15h apresenta a maior concentração de compras, indicando uma faixa prioritária para otimizar o atendimento e direcionar estratégias comerciais. Já os períodos de menor movimento representam oportunidades para ações promocionais que estimulem as vendas.

### 3.2 Produtos mais vendidos

o comando `nlargest()` serve para encontrar os maiores valores de uma tabela ou coluna, mostrando os registros com os valores mais altos

In [ ]:
df.nlargest(10, 'Total')

In [ ]:
top_quantidade = vendas.groupby('Description')['Quantity'].sum().nlargest(10)
top_receita = vendas.groupby('Description')['Total'].sum().nlargest(10)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
top_quantidade.sort_values().plot(kind='barh', ax=ax[0], color='steelblue')
ax[0].set_title('Top 10 produtos por quantidade')
top_receita.sort_values().plot(kind='barh', ax=ax[1], color='seagreen')
ax[1].set_title('Top 10 produtos por receita (£)')
plt.tight_layout()
plt.show()

> **Atenção:** o produto "PAPER CRAFT , LITTLE BIRDIE" aparece no topo, mas esse pedido de 80.995 unidades **foi cancelado logo depois**. Sempre desconfie de valores muito altos!

In [ ]:
# Quanto cada país representa na receita?
receita_pais = vendas.groupby('Country')['Total'].sum().sort_values(ascending=False)
participacao = receita_pais / receita_pais.sum() * 100
participacao.head(10)

In [ ]:
top_paises = (
    df.groupby('Country')['Total']
    .sum()
    .nlargest(10)
)

top_paises.sort_values().plot(
    kind='barh',
    figsize=(10, 5),
    color='teal'
)

plt.title('Top 10 países por receita (£)')
plt.xlabel('Receita (£)')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### 3.3 Produtos comprados juntos (Market Basket Analysis)

### Análise de Cestas de Compras com Apriori

O objetivo desta análise é identificar quais produtos costumam ser comprados juntos em um mesmo pedido, utilizando o algoritmo **Apriori** e as regras de associação.

**Etapas da análise:**

1. **Organização dos dados:** criamos uma tabela em que cada linha representa um pedido e cada coluna corresponde a um produto, indicando com `True` ou `False` se ele está presente na compra.
2. **Identificação de combinações frequentes:** aplicamos o algoritmo Apriori para encontrar conjuntos de produtos que aparecem juntos com frequência.
3. **Geração de regras de associação:** identificamos relações entre produtos, como a tendência de clientes que compram determinado item também adquirirem outro.

**Principais métricas das regras:**

- **Suporte (support):** percentual de pedidos em que os produtos aparecem juntos.
- **Confiança (confidence):** percentual de clientes que também compram o produto B entre aqueles que compraram o produto A.
- **Lift:** mede a força da associação entre dois produtos. Valores acima de 1 indicam uma associação positiva, ou seja, a compra de A está associada a uma maior probabilidade de compra de B.

**Recorte dos dados:** para reduzir o volume de processamento e evitar sobrecarregar o computador, a análise considera apenas os pedidos realizados no Reino Unido.

In [ ]:
vendas_uk = vendas[vendas['Country'] == 'United Kingdom']

cesta = vendas_uk.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack(fill_value=0)
cesta = cesta > 0                               # True se o produto está no pedido

# Mantém só produtos que aparecem em pelo menos 2% dos pedidos (deixa mais leve)
cesta = cesta.loc[:, cesta.mean() >= 0.02]
print('Pedidos x produtos:', cesta.shape)

In [ ]:
df_uk = df[df['Country'] == 'United Kingdom'].copy()

# Remover registros sem número de pedido ou produto
df_uk = df_uk.dropna(subset=['InvoiceNo', 'Description'])

# Criar a matriz de pedidos e produtos
cesta = (
    df_uk.groupby(['InvoiceNo', 'Description'])['Description']
    .count()
    .unstack(fill_value=0)
)

# Transformar quantidades em presença ou ausência
cesta = cesta > 0

cesta.head()

In [ ]:
itens_frequentes = apriori(cesta, min_support=0.02, use_colnames=True)
regras = association_rules(itens_frequentes, metric='lift', min_threshold=1)

# Ordena pelas regras mais fortes
regras = regras.sort_values('lift', ascending=False)

# Transforma os conjuntos em texto simples, para ficar fácil de ler
regras['Se comprou'] = regras['antecedents'].apply(lambda x: ', '.join(x))
regras['Também compra'] = regras['consequents'].apply(lambda x: ', '.join(x))

regras[['Se comprou', 'Também compra', 'support', 'confidence', 'lift']].head(10)

Essa análise permite identificar quais produtos são frequentemente comprados juntos, fornecendo informações estratégicas para desenvolver ações de venda conjunta, criar kits promocionais e recomendar produtos complementares, com o objetivo de aumentar o valor médio dos pedidos e impulsionar o faturamento.


### 3.4 Preços e vendas

Vamos ver como as vendas se dividem por **faixa de preço**. A função `pd.cut` separa os preços em faixas.

In [ ]:
faixas = [0, 1, 2, 3, 5, 10, 100000]
nomes = ['até £1', '£1-2', '£2-3', '£3-5', '£5-10', 'mais de £10']
vendas_faixa = vendas.copy()
vendas_faixa['FaixaPreco'] = pd.cut(vendas_faixa['UnitPrice'], bins=faixas, labels=nomes)

tabela = vendas_faixa.groupby('FaixaPreco', observed=True)[['Quantity', 'Total']].sum()
tabela_pct = tabela / tabela.sum() * 100
tabela_pct.columns = ['% das unidades', '% da receita']

tabela_pct.plot(kind='bar', figsize=(10, 4), color=['steelblue', 'coral'])
plt.title('Unidades e receita por faixa de preço')
plt.ylabel('%')
plt.xticks(rotation=0)
plt.show()
tabela_pct

**Análise do gráfico:** produtos com preço de até £2 representam aproximadamente 71% das unidades vendidas, mas correspondem a apenas 35,6% da receita. Em contrapartida, produtos acima de £2 representam uma parcela menor das unidades, mas contribuem proporcionalmente mais para o faturamento. Destaca-se a faixa de £3 a £5, que corresponde a cerca de 9,2% das unidades e gera 19,6% da receita. Esses resultados indicam que os produtos de menor preço impulsionam o volume de vendas, enquanto os itens mais caros têm maior participação na geração de receita.

In [ ]:
precos = vendas.copy()

# Preço mais comum de cada produto
preco_normal = precos.groupby('StockCode')['UnitPrice'].transform('median')
precos['PrecoRelativo'] = precos['UnitPrice'] / preco_normal * 100

# Separa as compras por quantidade de unidades
precos['FaixaQtd'] = pd.cut(precos['Quantity'], bins=[0, 5, 11, 23, 47, 99, 100000],
                            labels=['1-5', '6-11', '12-23', '24-47', '48-99', '100+'])

preco_por_qtd = precos.groupby('FaixaQtd', observed=True)['PrecoRelativo'].mean()

preco_por_qtd.plot(kind='bar', figsize=(9, 4), color='seagreen')
plt.axhline(100, color='gray', linestyle='--')
plt.title('Preço relativo pago por quantidade comprada (100 = preço normal)')
plt.xlabel('Unidades compradas na linha do pedido')
plt.xticks(rotation=0)
plt.show()
preco_por_qtd

**Análise do gráfico:** observa-se uma tendência de redução do preço relativo conforme aumenta a quantidade comprada por pedido. Nas compras de 1 a 5 unidades, o preço relativo médio é de aproximadamente 148, indicando valores cerca de 48% acima da mediana dos respectivos produtos. Já nas compras de 100 unidades ou mais, o índice cai para aproximadamente 89, sugerindo preços cerca de 11% abaixo da mediana. Esse comportamento indica uma possível relação entre maiores quantidades compradas e preços unitários menores. Entretanto, essa relação não comprova a existência de descontos por volume, pois outros fatores também podem influenciar os preços praticados.

### 3.5 Cancelamentos

In [ ]:
# Taxa de cancelamento por mês
faturas_mes = df.groupby('AnoMes')['InvoiceNo'].nunique()
canceladas_mes = cancelamentos.groupby('AnoMes')['InvoiceNo'].nunique()
taxa_mes = canceladas_mes / faturas_mes * 100

taxa_mes.plot(kind='bar', figsize=(12, 4), color='teal')
plt.title('Taxa de cancelamento por mês (% das faturas)')
plt.ylabel('%')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Os 10 maiores cancelamentos (linhas com mais unidades canceladas)
cancelamentos.sort_values('Quantity').head(10)[['InvoiceNo', 'Description', 'Quantity', 'Total', 'CustomerID', 'InvoiceDate']]

In [ ]:
# Taxa de cancelamento por país (só países com pelo menos 50 faturas)
faturas_pais = df.groupby('Country')['InvoiceNo'].nunique()
canceladas_pais = cancelamentos.groupby('Country')['InvoiceNo'].nunique()

tabela_pais = pd.DataFrame({'faturas': faturas_pais, 'canceladas': canceladas_pais}).fillna(0)
tabela_pais = tabela_pais[tabela_pais['faturas'] >= 50]
tabela_pais['taxa_%'] = tabela_pais['canceladas'] / tabela_pais['faturas'] * 100
tabela_pais.sort_values('taxa_%', ascending=False).head(10)

In [ ]:
# Quem cancela? Comparamos clientes que já cancelaram com os que nunca cancelaram
gasto_cliente = vendas.groupby('CustomerID')['Total'].sum()
clientes_que_cancelam = cancelamentos['CustomerID'].dropna().unique()

comparacao = pd.DataFrame({'gasto_total': gasto_cliente})
comparacao['ja_cancelou'] = comparacao.index.isin(clientes_que_cancelam)
comparacao.groupby('ja_cancelou')['gasto_total'].median()

**O que vemos sobre cancelamentos:**
- Cerca de **16% das faturas** são canceladas, e a taxa é parecida em todos os meses.
- Existem cancelamentos **gigantes** (80.995 e 74.215 unidades), que parecem **erro de digitação**. Uma confirmação para pedidos muito grandes evitaria isso.
- Quem cancela **gasta mais** (mediana bem maior). São clientes grandes **ajustando pedidos**, não clientes insatisfeitos.

## Passo 4: Segmentação de clientes

### 4.1 O que é RFM?

Para cada cliente calculamos 3 números:
- **R – Recência:** há quantos dias foi a última compra? (quanto **menor**, melhor)
- **F – Frequência:** quantos pedidos fez? (quanto **maior**, melhor)
- **M – Monetário:** quanto gastou no total? (quanto **maior**, melhor)

Usamos só vendas com cliente identificado.

In [ ]:
clientes = vendas.dropna(subset=['CustomerID'])

# Data de referência: o dia seguinte à última compra do arquivo
data_referencia = clientes['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Data de referência:', data_referencia.date())

rfm = clientes.groupby('CustomerID').agg(
    UltimaCompra=('InvoiceDate', 'max'),
    Frequencia=('InvoiceNo', 'nunique'),
    Monetario=('Total', 'sum')
)
rfm['Recencia'] = (data_referencia - rfm['UltimaCompra']).dt.days
rfm = rfm[['Recencia', 'Frequencia', 'Monetario']]

print('Número de clientes:', len(rfm))
rfm.head()

In [ ]:
rfm.describe()

### **4.2 Pré-processamento e Preparação dos Dados para o K-Means**

O algoritmo K-Means realiza o agrupamento de clientes (clustering) com base no cálculo de distâncias euclidianas. Para garantir a convergência e o desempenho ideal do modelo, foram aplicadas duas etapas cruciais de transformação nos dados:

**Transformação Logarítmica**: Devido à presença de outliers na variável de faturamento (clientes com volume de gastos desproporcionalmente elevado), aplicou-se a função logarítmica `np.log1p` para comprimir a amplitude dos valores extremos e aproximar a distribuição da variável a uma curva normal.

**Padronização das Variáveis (StandardScaler):** Como as métricas operam em grandezas distintas (ex.: valores monetários em milhares de libras versus contagem de frequência de pedidos), utilizou-se o `StandardScaler` para reescalar todas as variáveis para uma mesma escala padrão (média $0$ e desvio padrão $1$), impedindo que atributos com maior magnitude dominem o cálculo de distância do algoritmo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(rfm['Monetario'], bins=50, ax=ax[0])
ax[0].set_title('Monetário ANTES do log (muito concentrado)')
sns.histplot(np.log1p(rfm['Monetario']), bins=50, ax=ax[1], color='seagreen')
ax[1].set_title('Monetário DEPOIS do log (bem distribuído)')
plt.show()

In [ ]:
rfm_log = np.log1p(rfm)                 # log1p = log(1 + x), funciona mesmo com zero

scaler = StandardScaler()
rfm_padronizado = scaler.fit_transform(rfm_log)

print('Exemplo dos dados padronizados (5 primeiros clientes):')
print(rfm_padronizado[:5].round(2))

### **4.3 Definição do Número de Grupos (Clusters) ($k$)**

O algoritmo K-Means exige a definição prévia da quantidade de agrupamentos ($k$) a serem gerados. Para determinar a partição ideal da base de dados, avaliaram-se cenários no intervalo de $k = 2$ a $k = 10$, utilizando duas métricas complementares de validação de clustering:

**Método do Cotovelo (Elbow Method / WCSS)**: Avalia a Soma dos Quadrados Intra-cluster (Within-Cluster Sum of Squares — WCSS), que mede o grau de dispersão dos dados em relação ao centroide do seu grupo. O valor ideal de $k$ é identificado no ponto de inflexão da curva ("cotovelo"), a partir do qual o ganho em compactação torna-se marginal.

**Coeficiente de Silhueta (Silhouette Score)**: Mensura a coesão interna e a separação inter-cluster, variando no intervalo de $-1$ a $1$. Quanto mais próximo de $1$, melhor estruturada e bem delimitada está a separação entre os grupos.

In [ ]:
wcss = []
silhueta = []
valores_k = range(2, 11)

for k in valores_k:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    modelo.fit(rfm_padronizado)
    wcss.append(modelo.inertia_)
    silhueta.append(silhouette_score(rfm_padronizado, modelo.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(valores_k, wcss, marker='o')
ax[0].set_title('Método do cotovelo')
ax[0].set_xlabel('Número de grupos (k)')
ax[1].plot(valores_k, silhueta, marker='o', color='coral')
ax[1].set_title('Silhueta')
ax[1].set_xlabel('Número de grupos (k)')
plt.show()

A escolha de $k = 4$ fundamenta-se no equilíbrio entre o Método do Cotovelo e o Coeficiente de Silhueta: o gráfico do cotovelo indica uma desaceleração acentuada do erro a partir de $k=4$, enquanto a silhueta mantém uma qualidade de separação estável ($\approx 0,34$). Embora $k=2$ apresente maior silhueta, dividi-los em apenas dois grupos traz pouco valor prático para o negócio, tornando a solução com 4 clusters a mais adequada para estratégias diretas de marketing e retenção

### 4.4 Criar os grupos

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm['Grupo'] = kmeans.fit_predict(rfm_padronizado)

rfm['Grupo'].value_counts()

### 4.5 Entender cada grupo

Calculamos a média de R, F e M de cada grupo e quanto cada um representa em clientes e receita.

In [ ]:
perfil = rfm.groupby('Grupo').agg(
    Clientes=('Recencia', 'count'),
    Recencia_media=('Recencia', 'mean'),
    Frequencia_media=('Frequencia', 'mean'),
    Gasto_medio=('Monetario', 'mean'),
    Receita_total=('Monetario', 'sum')
)
perfil['% clientes'] = perfil['Clientes'] / perfil['Clientes'].sum() * 100
perfil['% receita'] = perfil['Receita_total'] / perfil['Receita_total'].sum() * 100

perfil.sort_values('Gasto_medio', ascending=False)

**Dando nome aos grupos**

Os números dos grupos (0, 1, 2, 3) são aleatórios e podem mudar de computador para computador. Por isso damos os nomes olhando o **perfil**:
- **Clientes de Alto Valor (High-Value):** recência baixa, frequência e gasto altos
- **Clientes Recorrentes em Declínio:** frequência e gasto médios, mas estão há um tempo sem comprar
- **Novos Compradores/Potenciais:** compraram há pouco tempo, mas poucas vezes
- **Inativos / Perdidos:** recência alta (sumiram) e gasto baixo

O código abaixo faz isso automaticamente.

In [ ]:
# 1) O grupo com maior gasto médio é o VIP; o de menor gasto médio é o Inativo
ordem = perfil.sort_values('Gasto_medio', ascending=False).index.tolist()
vip, meio1, meio2, inativo = ordem

# 2) Dos dois grupos do meio, o que comprou mais recentemente é o "Recentes"
if perfil.loc[meio1, 'Recencia_media'] < perfil.loc[meio2, 'Recencia_media']:
    recentes, em_risco = meio1, meio2
else:
    recentes, em_risco = meio2, meio1

nomes_grupos = {vip: 'Clientes de Alto Valor (High-Value)', em_risco: 'Clientes Recorrentes em Declínio',
                recentes: 'Novos Compradores/Potenciais', inativo: 'Inativos / Perdidos'}

rfm['Segmento'] = rfm['Grupo'].map(nomes_grupos)
perfil.index = perfil.index.map(nomes_grupos)
perfil.round(1)

### 4.6 Visualizar os grupos

In [ ]:
cores = {'Clientes de Alto Valor (High-Value)': 'tab:blue', 'Clientes Recorrentes em Declínio': 'tab:orange',
         'Novos Compradores/Potenciais': 'tab:green', 'Inativos / Perdidos': 'tab:gray'}

plt.figure(figsize=(9, 6))
sns.scatterplot(data=rfm, x='Recencia', y='Frequencia', hue='Segmento',
                palette=cores, alpha=0.6, s=20)
plt.yscale('log')            # escala log para enxergar melhor
plt.title('Clientes: recência x frequência')
plt.xlabel('Recência (dias desde a última compra)')
plt.ylabel('Frequência (nº de pedidos, escala log)')
plt.show()

In [ ]:
perfil[['% clientes', '% receita']].plot(kind='bar', figsize=(10, 4), color=['gray', 'goldenrod'])
plt.title('Cada grupo: % dos clientes x % da receita')
plt.xticks(rotation=15)
plt.show()

**O que vemos:** o grupo **Clientes de Alto Valor (High-Value)	** tem poucos clientes (cerca de 16%), mas gera a **maior parte da receita** (cerca de 65%). Os **Inativos** são muitos (cerca de 37%), mas trazem pouca receita.

### 4.7 Estratégia para cada grupo

| Grupo | Estratégia |
|---|---|
| **Clientes de Alto Valor (High-Value)** | Implantação de programa de fidelidade, isenção de frete, acesso antecipado às coleções sazonais (Natal) e atendimento VIP exclusivo. |
| **Clientes Recorrentes em Declínio** | Campanhas de Reconquista de Clientes via e-mail marketing personalizadas com base no histórico de compras, aliadas a cupons com gatilhos de escassez (prazo limitado). |
| **Novos Compradores/Potenciais** | Estruturação de fluxo de boas-vindas (onboarding), incentivo à segunda conversão e algoritmos de recomendação no formato "compre junto". |
| **Inativos / Perdidos** | Disparos automatizados e escaláveis em datas comemorativas de alta relevância (como liquidações e Natal), minimizando o custo de aquisição/reativação. |

In [ ]:
# Salva a lista de clientes com seu segmento (útil para o time de marketing)
rfm.to_csv('clientes_segmentados.csv')
print('Arquivo clientes_segmentados.csv salvo!')

# **Passo 5: Conclusões e recomendações**

### **5.1 Conclusões:**

**Comportamento de Compra:**  Foco em clientes empresariais/revendedores, com pedidos volumosos concentrados no horário comercial e volume operacional reduzido aos domingos.

**Sazonalidade e Geografia:** Concentração de vendas no Reino Unido e dependência do quadrimestre final do ano (pico entre setembro e novembro).

**Distribuição de Receita:** Forte assimetria na base, com $16\%$ dos clientes de alto valor a responderem por $65\%$ do faturamento total.

**Padrões e Cancelamentos:** Taxa de $16\%$ de cancelamentos em faturas (ajustes de grandes pedidos e erros de digitação) e preferência por aquisição de coleções completas.

### **5.2 Plano de Ação:**

**Recomendado Controle de Ruído Operacional**: Implementar checagem automática para pedidos com volumes atípicos para estancar os $16\%$ de cancelamentos.

**Maximização de LTV (Clientes VIP)**: Oferecer benefícios exclusivos e catálogo antecipado de Natal para a fatia de $16\%$ dos clientes que garante $65\%$ da receita.

**Aumento do Ticket Médio:** }Utilizar algoritmos de recomendação baseados nas coleções (cross-sell) e descontos por lote/quantidade (up-sell).


**Aproveitamento Sazonal:** Disparar campanhas de win-back para reativar clientes em declínio no terceiro trimestre, garantindo tração máxima antes do pico de vendas.